# Structured Meal & Grocery Planner with CrewAI

A multi-agent planner that turns a few meal ideas into a structured weekly grocery run.

Four agents work in sequence: a **meal planner** designs the meals, a **shopping organizer** groups ingredients by store section, a **budget advisor** estimates cost, and a **leftover specialist** suggests ways to use what's left over.

The output is typed with Pydantic models rather than free text, so the plan comes back as structured data you can act on.

## Setup


Libraries: `crewai` and `crewai-tools` for the agents and search, `pydantic` for the output schemas, `python-dotenv` for keys.

In [ ]:
%%capture
%pip install crewai crewai-tools python-dotenv
%pip install langchain langchain-community

`leftover.py` and the `config/` YAML files ship with this project — no download needed.

In [ ]:
import sys

sys.path.append(".")

Check the local `LeftoversCrew` imports.

In [ ]:
from leftover import LeftoversCrew

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Optional
from IPython.display import display, JSON, Markdown
from datetime import datetime
import os

## Structured output with Pydantic

Typed models make the crew return structured data instead of prose, so downstream code can read fields directly.

### `GroceryItem` — one item, with quantity and category.

In [ ]:
class GroceryItem(BaseModel):
    """Individual grocery item"""
    name: str = Field(description="Name of the grocery item")
    quantity: str = Field(description="Quantity needed (for example, '2 lbs', '1 gallon')")
    estimated_price: str = Field(description="Estimated price (for example, '$3-5')")
    category: str = Field(description="Store section (for example, 'Produce', 'Dairy')")

Create one.

In [ ]:
sample_item = GroceryItem(
    name="Chicken Breast",
    quantity="2 lbs",
    estimated_price="$8-12",
    category="Meat"
)

It behaves like a JSON object.

In [ ]:
sample_item

In [ ]:
type(sample_item)

`display` renders it more readably.

In [ ]:
# Display structured data
print("🛒 Sample Grocery Item Structure:")
display(JSON(sample_item.model_dump()))

### `MealPlan` — a meal and the items it needs.

In [ ]:
class MealPlan(BaseModel):
    """Simple meal plan"""
    meal_name: str = Field(description="Name of the meal")
    difficulty_level: str = Field(description="'Easy', 'Medium', 'Hard'")
    servings: int = Field(description="Number of people it serves")
    researched_ingredients: List[str] = Field(description="Ingredients found through research")

Create one.

In [ ]:
sample_meal = MealPlan(
    meal_name="Chicken Stir Fry",
    difficulty_level="Easy",
    servings=4,
    researched_ingredients=["chicken breast", "broccoli", "bell peppers", "garlic", "soy sauce", "rice"]
)

In [ ]:
sample_meal

In [ ]:
print("\n🍽️ Sample Meal Plan Structure:")
display(JSON(sample_meal.model_dump()))

### `ShoppingCategory` — items grouped by store section.

In [ ]:
class ShoppingCategory(BaseModel):
    """Store section with items"""
    section_name: str = Field(description="Store section (for example, 'Produce', 'Dairy')")
    items: List[GroceryItem] = Field(description="Items in this section")
    estimated_total: str = Field(description="Estimated cost for this section")

Create one.

In [ ]:
sample_section = ShoppingCategory(
    section_name="Produce",
    items=[
        GroceryItem(name="Bell Peppers", quantity="3 pieces", estimated_price="$3-4", category="Produce"),
        GroceryItem(name="Onions", quantity="2 lbs", estimated_price="$2-3", category="Produce")
    ],
    estimated_total="$5-7"
)

In [ ]:
print("\n🏪 Sample Shopping Section:")
display(JSON(sample_section.model_dump()))

### `GroceryShoppingPlan` — the full plan the crew returns.

In [ ]:
class GroceryShoppingPlan(BaseModel):
    """Complete simplified shopping plan"""
    total_budget: str = Field(description="Total planned budget")
    meal_plans: List[MealPlan] = Field(description="Planned meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections")
    shopping_tips: List[str] = Field(description="Money-saving and efficiency tips")

Assemble a complete example.

## The model and tools

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

# Load GROQ_API_KEY (and SERPER_API_KEY where needed) from a .env file.
load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is not set. Copy .env.example to .env and add your key "
        "(free at https://console.groq.com/keys)."
    )

# Groq through its OpenAI-compatible endpoint.
llm = LLM(
    model="openai/llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens=2000,
)

Web search, for looking up recipes and prices.

In [ ]:
# Serper-backed web search (SERPER_API_KEY is loaded above).
search_tool = SerperDevTool()

## Agents and tasks

### Meal planner

Designs meals from the user's request.

In [ ]:
meal_planner = Agent(
    role="Meal Planner & Recipe Researcher",
    goal="Search for optimal recipes and create detailed meal plans",
    backstory="A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

Its task, typed with `MealPlan`.

In [ ]:
meal_planning_task = Task(
    description=(
        "Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. "
        "Find recipes that match the skill level and provide complete ingredient lists with quantities."
    ),
    expected_output="A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.",
    agent=meal_planner,
    output_pydantic=MealPlan,
    output_file="meals.json"
)

Run the meal-planning crew on its own first.

In [ ]:
from crewai import Crew, Process

meal_planner_crew = Crew(
    agents=[meal_planner],
    tasks=[meal_planning_task],
    process=Process.sequential,  # Ensures tasks are executed in order
    verbose=True
)

meal_planner_result = meal_planner_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],       
        "cooking_skill": "beginner"                
    }
)
print("✅ Single meal planning completed!")
print("📋 Single Meal Results:")
print(meal_planner_result)

### Shopping organizer

Groups the ingredients by store section.

In [ ]:
shopping_organizer = Agent(
    role="Shopping Organizer", 
    goal="Organize grocery lists by store sections efficiently",
    backstory="An experienced shopper who knows how to organize lists for quick store trips and considers dietary restrictions.",
    tools=[],
    llm=llm,
    verbose=False
)

Its task.

In [ ]:
shopping_task = Task(
    description=(
        "Organize the ingredients from the '{meal_name}' meal plan into a grocery shopping list. "
        "Group items by store sections and estimate quantities for {servings} people. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill: {cooking_skill}. "
        "Stay within budget: {budget}."
    ),
    expected_output="An organized shopping list grouped by store sections with quantities and prices.",
    agent=shopping_organizer,
    context=[meal_planning_task],
    output_pydantic=GroceryShoppingPlan,
    output_file="shopping_list.json"
)

Both agents as one crew, running sequentially.

In [ ]:
two_agent_grocery_crew = Crew(
    agents=[meal_planner, shopping_organizer],  # Both agents
    tasks=[meal_planning_task, shopping_task],   # Both tasks
    process=Process.sequential,
    verbose=True
)

# Run the complete crew (this will do BOTH meal planning AND shopping)
shopping_result = two_agent_grocery_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],      
        "cooking_skill": "beginner"               
    }
)

# Print the shopping results
print("✅ Complete meal planning + shopping completed!")
print("📋 Shopping Results:")
print(shopping_result)

### Budget advisor

Estimates cost and flags savings.

In [ ]:
budget_advisor = Agent(
    role="Budget Advisor",
    goal="Provide cost estimates and money-saving tips",
    backstory="A budget-conscious shopper who helps families save money on groceries while respecting dietary needs.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

Its task.

In [ ]:
budget_task = Task(
    description=(
        "Analyze the shopping plan for '{meal_name}' serving {servings} people. "
        "Ensure total cost stays within {budget}. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide practical money-saving tips and alternative ingredients if needed to meet budget."
    ),
    expected_output="A complete shopping guide with detailed prices, budget analysis, and money-saving tips.",
    agent=budget_advisor,
    context=[meal_planning_task, shopping_task],
    output_file="shopping_guide.md"
)

### Config-driven agents with YAML

Agents and tasks can also be declared in YAML instead of Python. `leftover.py` uses `@CrewBase` to load `config/agents.yaml` and `config/tasks.yaml`.

`@CrewBase` wires the YAML definitions to methods decorated with `@agent` and `@task`, which keeps prompt text out of the code.

The local `LeftoversCrew`:

In [ ]:
from leftover import LeftoversCrew

leftovers_cb = LeftoversCrew(llm=llm)
yaml_leftover_manager = leftovers_cb.leftover_manager()
yaml_leftover_task    = leftovers_cb.leftover_task()

### Summary agent

Pulls the whole plan together.

In [ ]:
summary_agent = Agent(
    role="Report Compiler",
    goal="Compile comprehensive meal planning reports from all team outputs",
    backstory="A skilled coordinator who organizes information from multiple specialists into comprehensive, easy-to-follow reports.",
    tools=[],
    llm=llm,
    verbose=False
)

In [ ]:
summary_task = Task(
    description=(
        "Compile a comprehensive meal planning report that includes:\n"
        "1. The complete recipe and cooking instructions from the meal planner\n"
        "2. The organized shopping list with prices from the shopping organizer\n"
        "3. The budget analysis and money-saving tips from the budget advisor\n"
        "4. The leftover management suggestions from the waste reduction specialist\n"
        "Format this as a complete, user-friendly meal planning guide."
    ),
    expected_output="A comprehensive meal planning guide that combines all team outputs into one cohesive report.",
    agent=summary_agent,
    context=[meal_planning_task, shopping_task, budget_task, yaml_leftover_task],
)

### The complete crew

All agents and tasks in one sequential pipeline.

In [ ]:
complete_grocery_crew = Crew(
    agents=[
        meal_planner,           
        shopping_organizer,      
        budget_advisor,         
        yaml_leftover_manager,  # YAML-based leftover manager
        summary_agent           # New summary agent
    ],
    tasks=[
        meal_planning_task,     
        shopping_task,          
        budget_task,            
        yaml_leftover_task,    # YAML-based leftover task
        summary_task            # New summary task
    ],
    process=Process.sequential,
    verbose=True
)

Run it.

In [ ]:
# Run the complete crew
complete_result = complete_grocery_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",
        "dietary_restrictions": ["no nuts", "low sodium"],
        "cooking_skill": "beginner"
    }
)

print("✅ Complete meal planning with summary completed!")
print("📋 Complete Results:")
print(complete_result)

## Extending the crew

### A dietary-requirements agent

Adapts the plan for allergies and dietary restrictions.

In [ ]:
nutrition_analyst = Agent(
    role="",  # TODO: Define the role
    goal="",  # TODO: Define the goal  
    backstory="",  # TODO: Create backstory
    tools=[],  # TODO: Add appropriate tools
    llm=llm,
    verbose=False
)


### Extending the schemas for weekly planning

In [ ]:
from enum import Enum
from typing import Dict

class MealType(str, Enum):
    # Add meal types here
    pass

class WeeklyMealPlan(BaseModel):
    # Add fields for weekly planning
    pass

class WeeklyGroceryPlan(BaseModel):
    # Extend GroceryShoppingPlan for weekly use
    pass


## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)